# Full-text search: a hands-on implementation

**Track:** Search · **Guide:** [`043-full-text-search.md`](../guides/concepts/043-full-text-search.md)

Full-text search tokenizes text, applies language-aware processing, and matches query terms against an inverted index.

> **Intuition:** A book index maps each word to the pages on which it appears.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Full-text search]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Search product manuals for exact error codes and domain terms before adding more advanced retrieval.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Build an inverted index

An inverted index maps each normalized token to the documents containing it.


In [ ]:
import re
from collections import defaultdict

documents = {
    "manual-1": "Reset the account password from account settings",
    "manual-2": "If error E1042 appears, restart the device",
    "manual-3": "Contact support to restore a locked account",
}
index = defaultdict(set)
for document_id, text in documents.items():
    for token in set(re.findall(r"[a-z0-9]+", text.lower())):
        index[token].add(document_id)
print("indexed terms:", len(index))

## 2. Search with AND and OR semantics

Intersect postings for AND; union postings for OR. Real search engines add stemming, ranking, syntax, and language processing.


In [ ]:
def search(query, mode="AND"):
    terms = re.findall(r"[a-z0-9]+", query.lower())
    if not terms:
        return set()
    postings = [index.get(term, set()) for term in terms]
    if mode == "AND":
        return set.intersection(*postings)
    if mode == "OR":
        return set.union(*postings)
    raise ValueError("mode must be AND or OR")

print("AND:", search("account password"))
print("OR:", search("account E1042", mode="OR"))

## 3. Rank matching documents

A simple overlap score helps order results. Use BM25 or a production search engine when collection size and relevance requirements grow.


In [ ]:
def ranked_search(query):
    terms = set(re.findall(r"[a-z0-9]+", query.lower()))
    scores = {
        document_id: sum(document_id in index[term] for term in terms)
        for document_id in documents
    }
    return sorted(
        ((score, document_id) for document_id, score in scores.items() if score),
        reverse=True,
    )

print(ranked_search("account password"))

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/043-full-text-search.md) and the
[learning roadmap](../guides/ROADMAP.md).
